In [12]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
print("✅ Working directory set to project root:", PROJECT_ROOT)


✅ Working directory set to project root: /home/robert/Projects/Feb-Hackathon


In [13]:
import pandas as pd

In [14]:
DATAFOLDER = PROJECT_ROOT / "data"
RAW_FOLDER = DATAFOLDER / "raw"
EXTRACTED_FOLDER = DATAFOLDER / "extracted"
RAW_FOLDER.mkdir(parents=True, exist_ok=True)
EXTRACTED_FOLDER.mkdir(parents=True, exist_ok=True)
print("✅ Data folders set up at:", DATAFOLDER)

✅ Data folders set up at: /home/robert/Projects/Feb-Hackathon/data


In [15]:
FILES = {
    "2025_Q1": RAW_FOLDER / "personal_wellbeing_loneliness_q1_2025.xlsx",
    "2025_Q2": RAW_FOLDER / "personal_wellbeing_loneliness_q2_2025.xlsx",
    "2025_Q3": RAW_FOLDER / "personal_wellbeing_loneliness_q3_2025.xlsx",
    "2025_Q4": RAW_FOLDER / "personal_wellbeing_loneliness_q4_2025.xlsx",
}

In [16]:
LONELY_CATS = [
    "Often or always",
    "Some of the time",
    "Occasionally",
    "Hardly ever",
    "Never",
]

In [17]:
CAT_RENAME = {
    "Often or always": "often",
    "Some of the time": "some_time",
    "Occasionally": "occasionally",
    "Hardly ever": "hardly_ever",
    "Never": "never",
}

In [18]:
SKIPROWS = 10  # keep consistent with what you used

In [19]:
def extract_loneliness_quarter(path: Path, quarter: int) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name="Table_11", skiprows=SKIPROWS)

    # Keep Region + Country + All persons
    df = df[df["Breakdown category"].isin(
        ["Region", "Country", "All persons"])].copy()

    # Rename national row
    df.loc[df["Breakdown category"] ==
           "All persons", "Breakdown"] = "Great Britain"

    # Keep only substantive response categories
    df = df[df["Response options"].isin(LONELY_CATS)].copy()

    # Clean + numeric
    df["Breakdown"] = df["Breakdown"].astype(str).str.strip()
    df["Response options"] = df["Response options"].astype(str).str.strip()
    df["Estimate (%)"] = pd.to_numeric(df["Estimate (%)"],
                                       errors="coerce")  # handles [low] etc

    # Add geography_level (normalised)
    df["geography_level"] = df["Breakdown category"].replace({
        "Region": "region",
        "Country": "country",
        "All persons": "national",
    })

    # Wide pivot: one row per geography (region/country/national)
    wide = (
        df.pivot_table(
            index=["geography_level", "Breakdown"],
            columns="Response options",
            values="Estimate (%)",
            aggfunc="first"
        )
        .reset_index()
    )

    # Rename columns
    wide = wide.rename(columns={"Breakdown": "geography", **CAT_RENAME})

    # Ensure all 5 columns exist (just in case)
    for col in CAT_RENAME.values():
        if col not in wide.columns:
            wide[col] = pd.NA

    # Add quarter + headline loneliness metrics
    wide["quarter"] = quarter

    wide["lonely_at_least_occasionally_pct"] = (
        wide["often"].fillna(0)
        + wide["some_time"].fillna(0)
        + wide["occasionally"].fillna(0)
    ).round(0).astype(int)

    wide["lonely_often_or_sometimes_pct"] = (
        wide["often"].fillna(0) + wide["some_time"].fillna(0)
    ).round(0).astype(int)

    # Reorder
    wide = wide[
        ["quarter", "geography_level", "geography",
         "often", "some_time", "occasionally", "hardly_ever", "never",
         "lonely_at_least_occasionally_pct", "lonely_often_or_sometimes_pct"]
    ]

    # Sanity checks (should now be 13 per quarter)
    if wide["geography"].nunique() != 13:
        print(
            f"⚠️ {quarter}: expected 13 geographies, got {wide['geography'].nunique()}")

    return wide

In [20]:
# Build all quarters
lonely_frames = [extract_loneliness_quarter(path, q) for q, path in FILES.items()]
final_loneliness = pd.concat(lonely_frames, ignore_index=True)

print(final_loneliness.shape)  # should be (36, 8)
print(final_loneliness.head())

# Save
OUT_PATH = EXTRACTED_FOLDER / "loneliness_quarterly.csv"
final_loneliness.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")

(52, 10)
Response options  quarter geography_level      geography  often  some_time  \
0                 2025_Q1         country        England      7         20   
1                 2025_Q1         country       Scotland      8         15   
2                 2025_Q1         country          Wales      6         15   
3                 2025_Q1        national  Great Britain      7         19   
4                 2025_Q1          region  East Midlands      6         21   

Response options  occasionally  hardly_ever  never  \
0                           24           29     18   
1                           22           33     20   
2                           30           28     18   
3                           24           29     19   
4                           25           26     20   

Response options  lonely_at_least_occasionally_pct  \
0                                               51   
1                                               45   
2                                  

In [22]:
final_loneliness.groupby("quarter")["geography"].nunique()
# should be 13 each quarter

quarter
2025_Q1    13
2025_Q2    13
2025_Q3    13
2025_Q4    13
Name: geography, dtype: int64

In [23]:

final_loneliness.isna().sum()

Response options
quarter                             0
geography_level                     0
geography                           0
often                               0
some_time                           0
occasionally                        0
hardly_ever                         0
never                               0
lonely_at_least_occasionally_pct    0
lonely_often_or_sometimes_pct       0
dtype: int64